In [0]:
%run ../01-ingestion/setup-storage-connection

In [0]:
bronze_customers = spark.read.format("parquet").load("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/customers/")
bronze_customers.show(5)
print(f"Bronze customer count: {bronze_customers.count()}")

In [0]:
bronze_customers.write.format("delta").mode("overwrite").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/")

In [0]:
dbutils.fs.ls("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/")

In [0]:
from pyspark.sql.functions import lit, current_date

silver_customers_initial = bronze_customers \
    .withColumn("effective_start_date", current_date()) \
    .withColumn("effective_end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True))

silver_customers_initial.show(5)

In [0]:
silver_customers_initial.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/")

In [0]:
dbutils.fs.ls("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/_delta_log/")

In [0]:
sample_row = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/").limit(1).collect()[0]
sample_customer_id = sample_row["customer_id"]
print(f"Testing with customer: {sample_customer_id}, current country: {sample_row['country']}")

In [0]:
updated_customer = spark.createDataFrame([{
    "customer_id": sample_customer_id,
    "first_name": sample_row["first_name"],
    "last_name": sample_row["last_name"],
    "email": sample_row["email"],
    "country": "Japan",
    "signup_date": sample_row["signup_date"]
}])

updated_customer.show()

In [0]:
from delta.tables import DeltaTable

silver_customers_table = DeltaTable.forPath(spark, "abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/")

In [0]:
from pyspark.sql.functions import current_date

silver_customers_table.update(
    condition=f"customer_id = '{sample_customer_id}' AND is_current = true",
    set={
        "is_current": "false",
        "effective_end_date": "current_date()"
    }
)

In [0]:
new_version = updated_customer \
    .withColumn("effective_start_date", current_date()) \
    .withColumn("effective_end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True))

new_version.write.format("delta").mode("append").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/")

In [0]:
result = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/")
result.filter(f"customer_id = '{sample_customer_id}'").show()

In [0]:
current_state = result.filter("is_current = true")
print(f"Current customers count: {current_state.count()}")

historical_state = result.filter("is_current = false")
print(f"Historical customers count: {historical_state.count()}")